![Redis](https://redis.io/wp-content/uploads/2024/04/Logotype.svg?auto=webp&quality=85,75&width=120)

# Featureform Feature Versioning with Variants

In this recipe we register **two variants of the same feature** in [**Featureform**](https://docs.featureform.com/), serve each independently from **Redis**, and see how variants let you evolve feature logic without breaking models pinned to the old definition.

## Why variants
Feature logic changes — you tighten a filter, switch an aggregation, fix a bug. If you edit a feature in place, every model that used the old values silently starts seeing new ones, and past predictions become impossible to reproduce. Featureform's answer is the **variant**: a named version of a feature. Old and new coexist; each model **pins** the variant it was trained on; the dashboard records both.

## What we'll build
One feature, `avg_transactions`, in two variants:
- **`v1`** — average of *all* transactions.
- **`v2`** — average of only *high-value* transactions (a later refinement).

Both are materialized to Redis and served side by side.

## The stack — all local, no Spark

- **ClickHouse** — offline store; runs a SQL transformation per variant.
- **Redis** — online store; serves both variants.
- **Featureform** coordinator.

> ⚠️ **Needs local Docker; will not run on Colab or in CI.** The cells below start everything this recipe needs: the two provider containers on a private Docker network, plus the Featureform coordinator (gRPC `localhost:7878`, dashboard `http://localhost` — [install docs](https://docs.featureform.com/deployment/quickstart-docker)).

### Start ClickHouse and Redis

Both run on a private Docker network (`ff-net`) so the Featureform coordinator can reach them by container name — no host-port collisions. Only ClickHouse's HTTP port is published (on `18123`) so the data-load cell below can connect.

In [1]:
# NBVAL_SKIP
# Launch this notebook's own containers on a private Docker network (ff-net) so the
# Featureform coordinator reaches them by name — no host-port collisions. Names are
# project-scoped (ff-clickhouse/ff-redis) so they won't clash with other containers on
# your machine. Only ClickHouse's HTTP port is published (on 18123, to avoid the common
# 8123) so this notebook can load data; the native port and Redis stay inside ff-net.
# ClickHouse is pinned to 24.10 — the native-protocol version the coordinator speaks.
!docker rm -f ff-clickhouse ff-redis 2>/dev/null
!docker network create ff-net 2>/dev/null || true
!docker run -d --network ff-net --name ff-clickhouse -p 18123:8123 -e CLICKHOUSE_PASSWORD=featureform clickhouse/clickhouse-server:24.10
!docker run -d --network ff-net --name ff-redis redis:8

57276e8a3ffb5f1dc7447ade156557d7cae1caf41a048c40592888e3fba24f74
46f387815f93d3f15ca50945ba5b6c8073d560fef1886e11e616a726c8ef6e80
3e44c948dae889136acf704fa158640f9ff65d188d7f1bfb68d0186110565387


## Environment Setup

### Install Python Dependencies

In [2]:
%pip install -q featureform redis clickhouse-connect numpy

Note: you may need to restart the kernel to use updated packages.


### Start the Featureform coordinator

Skip this cell if you already ran `featureform deploy docker` in a terminal. It starts the coordinator (gRPC `localhost:7878`, dashboard `http://localhost`) and attaches it to `ff-net` so it can reach ClickHouse and Redis by container name.

In [3]:
# NBVAL_SKIP
import sys
# Start the Featureform coordinator (gRPC on localhost:7878, dashboard on http://localhost).
# Invoke via the kernel's own interpreter so it works even if the `featureform` console
# script isn't on PATH (common in Jupyter/VSCode after %pip install). Then attach the
# coordinator to ff-net, retrying until confirmed, so it resolves ClickHouse/Redis by name.
!{sys.executable} -m featureform deploy docker
!for i in $(seq 10); do docker network connect ff-net featureform 2>/dev/null; docker inspect featureform --format '{{range $k,$v := .NetworkSettings.Networks}}{{$k}} {{end}}' 2>/dev/null | grep -q ff-net && echo "coordinator attached to ff-net" && break; sleep 1; done

Deploying Featureform on Docker
Starting Docker deployment on Darwin 24.6.0
Checking if featureform container exists...
	Container featureform not found. Creating new container...
	'featureform' container started

Featureform is now running!
To access the dashboard, visit http://localhost:80
To apply definition files, run `featureform apply <file.py> --host http://localhost:7878 --insecure`
coordinator attached to ff-net


### Configure connections

In [4]:
import os

# This recipe is *about* variants, so disable Featureform's "equivalent variant" auto-reuse.
# With it on (the default, FF_GET_EQUIVALENT_VARIANTS), applying two feature variants that
# differ only in their source transformation makes the coordinator judge v2 "equivalent" to
# v1 and collapse it ("equivalent feature variant already exists, going to use its variant:
# v1") — so v2 never registers and serving ("avg_transactions", "v2") fails with a metadata
# NotFound. Turning it off registers each variant on its own.
os.environ["FF_GET_EQUIVALENT_VARIANTS"] = "false"

# Featureform coordinator (gRPC)
FEATUREFORM_HOST = os.getenv("FEATUREFORM_HOST", "localhost:7878")

# The coordinator reaches the providers by container name over the shared ff-net network.
# ClickHouse offline store (recent images require a password for network access)
CLICKHOUSE_HOST = os.getenv("CLICKHOUSE_HOST", "ff-clickhouse")
CLICKHOUSE_NATIVE_PORT = int(os.getenv("CLICKHOUSE_NATIVE_PORT", "9000"))
CLICKHOUSE_HTTP_PORT = int(os.getenv("CLICKHOUSE_HTTP_PORT", "18123"))  # published to host for the data-load cell
CLICKHOUSE_USER = os.getenv("CLICKHOUSE_USER", "default")
CLICKHOUSE_PASSWORD = os.getenv("CLICKHOUSE_PASSWORD", "featureform")
CLICKHOUSE_DATABASE = os.getenv("CLICKHOUSE_DATABASE", "default")

# Redis online store
REDIS_HOST = os.getenv("REDIS_HOST", "ff-redis")
REDIS_PORT = int(os.getenv("REDIS_PORT", "6379"))
REDIS_PASSWORD = os.getenv("REDIS_PASSWORD", "")

### Create a sample transactions table in ClickHouse

In [5]:
# NBVAL_SKIP
import time
import clickhouse_connect
import numpy as np

# ClickHouse may still be starting up — retry until it accepts connections.
for _ in range(30):
    try:
        ch = clickhouse_connect.get_client(
            host="localhost", port=CLICKHOUSE_HTTP_PORT,
            username=CLICKHOUSE_USER, password=CLICKHOUSE_PASSWORD,
        )
        ch.command("SELECT 1")
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError(f"ClickHouse not reachable on localhost:{CLICKHOUSE_HTTP_PORT} — is the container running?")

ch.command("DROP TABLE IF EXISTS transactions")
ch.command(
    """
    CREATE TABLE transactions (
        TransactionID String,
        CustomerID String,
        TransactionAmount Float64
    ) ENGINE = MergeTree ORDER BY CustomerID
    """
)
rng = np.random.default_rng(42)
rows = [[f"T{i:05d}", f"C{int(rng.integers(1000, 1050)):04d}", round(float(rng.gamma(2.0, 50.0)), 2)]
        for i in range(500)]
ch.insert("transactions", rows, column_names=["TransactionID", "CustomerID", "TransactionAmount"])
print("rows:", ch.command("SELECT count() FROM transactions"))

rows: 500


## Register providers and source

In [6]:
import featureform as ff

clickhouse = ff.register_clickhouse(
    name="clickhouse-quickstart",
    description="ClickHouse offline store with transaction history",
    host=CLICKHOUSE_HOST,
    port=CLICKHOUSE_NATIVE_PORT,
    user=CLICKHOUSE_USER,
    password=CLICKHOUSE_PASSWORD,
    database=CLICKHOUSE_DATABASE,
)

redis = ff.register_redis(
    name="redis-quickstart",
    description="Redis online (inference) store",
    host=REDIS_HOST,
    port=REDIS_PORT,
    password=REDIS_PASSWORD,
    db=0,
)

In [7]:
transactions = clickhouse.register_table(
    name="transactions", variant="quickstart", table="transactions",
)

## Two transformations — the old logic and the refinement

Each variant is backed by its own transformation. `v1` averages every transaction; `v2` averages only transactions above 100. Both output the same shape (`user_id`, `avg_transaction_amt`) so they can be variants of one feature.

In [8]:
@clickhouse.sql_transformation(variant="v1")
def average_user_transaction_all():
    """v1: average of all transactions."""
    return (
        "SELECT CustomerID AS user_id, avg(TransactionAmount) AS avg_transaction_amt "
        "FROM {{transactions.quickstart}} GROUP BY CustomerID"
    )

@clickhouse.sql_transformation(variant="v2")
def average_user_transaction_highvalue():
    """v2: average of only high-value (>100) transactions."""
    return (
        "SELECT CustomerID AS user_id, avg(TransactionAmount) AS avg_transaction_amt "
        "FROM {{transactions.quickstart}} WHERE TransactionAmount > 100 GROUP BY CustomerID"
    )

## Declare the feature in two variants

`ff.Variants` groups multiple versions under a single feature name. Each entry points at its own transformation and carries its own `variant` tag. Both materialize to Redis, and neither can clobber the other.

In [9]:
@ff.entity
class User:
    avg_transactions = ff.Variants({
        "v1": ff.Feature(
            average_user_transaction_all[["user_id", "avg_transaction_amt"]],
            variant="v1",
            type=ff.Float32,
            inference_store=redis,
        ),
        "v2": ff.Feature(
            average_user_transaction_highvalue[["user_id", "avg_transaction_amt"]],
            variant="v2",
            type=ff.Float32,
            inference_store=redis,
        ),
    })

## Apply

Both transformations run and both variants materialize into Redis.

In [10]:
# NBVAL_SKIP
client = ff.Client(host=FEATUREFORM_HOST, insecure=True)
client.apply(asynchronous=False, verbose=True)

Applying Run: sleepy_almeida
Creating User default_owner 
Creating Provider clickhouse-quickstart 
Creating Provider redis-quickstart 
Creating Source Variant transactions quickstart
Creating Source Variant average_user_transaction_all v1
Creating Source Variant average_user_transaction_highvalue v2
Creating Entity user 
Creating Feature Variant avg_transactions v1
Creating Feature Variant avg_transactions v2



UserWarning: install "ipywidgets" for Jupyter support

## Serve each variant — pinned by name

A caller asks for `avg_transactions` **and a specific variant**. Same feature name, two definitions, two values. A model trained on `v1` keeps requesting `v1` and is completely unaffected by the later `v2`.

In [11]:
# NBVAL_SKIP
user_id = client.dataframe(average_user_transaction_all)["user_id"].iloc[0]

v1 = client.features([("avg_transactions", "v1")], {"user": user_id})
v2 = client.features([("avg_transactions", "v2")], {"user": user_id})
print(f"user {user_id}")
print(f"  avg_transactions v1 (all txns):        {v1}")
print(f"  avg_transactions v2 (high-value only): {v2}")

No resources to apply
user C1047
  avg_transactions v1 (all txns):        [92.00199890136719]
  avg_transactions v2 (high-value only): [152.7857208251953]


## Cleanup

Stop and remove the containers when you're done.

In [12]:
# NBVAL_SKIP
import sys
# Tear down everything this notebook started. Remove containers (including the coordinator)
# before the network; the coordinator's endpoint releases asynchronously, so retry the delete.
!{sys.executable} -m featureform stop docker
!docker rm -f ff-clickhouse ff-redis featureform 2>/dev/null
!for i in $(seq 5); do docker network rm ff-net 2>/dev/null && break || sleep 1; done

Tearing down Featureform on Docker
Stopping containers...
	Stopping featureform container
Container quickstart-clickhouse not found. Skipping...
ff-clickhouse
ff-redis
featureform
ff-net


## Learn more

- [Featureform variants & versioning](https://docs.featureform.com/)
- [Featureform transformations & lineage recipe](./03_featureform_transformations_lineage.ipynb)
- [Featureform + Redis fraud detection recipe](./02_featureform_fraud_detection.ipynb)